# 6장 반복측정 자료: 혼합모형과 GEE (R과 파이썬으로 하는 통계분석)

보건통계학 기초 10장의 예로 반복측정 분산분석과 구형성 보정, 넓은 형식과 긴 형식, 무작위 절편·기울기 모형, 공분산 구조와 MMRM, GEE와 로지스틱 혼합모형을 R로 계산합니다. Colab에서는 런타임 → 런타임 유형 변경에서 R을 고른 뒤 셀을 위에서부터 차례로 실행하세요. 패키지가 없으면 먼저 install.packages(c("tidyr", "dplyr", "afex", "lmerTest", "mmrm", "emmeans", "geepack"))를 실행합니다.


## a. 반복측정 분산분석

### 셀 1. 환자 4명 × 방문 3회의 반복측정 분산분석

In [ ]:
toy <- data.frame(id = factor(rep(1:4, each = 3)), visit = factor(rep(c(0, 3, 6), 4)),
                  hba1c = c(9.2, 9.0, 8.5,  7.4, 7.3, 6.9,  8.4, 7.8, 7.8,  10.2, 9.9, 9.6))
summary(aov(hba1c ~ visit + Error(id / visit), data = toy))   # 환자 간 변동을 떼어 낸 반복측정 분산분석
summary(aov(hba1c ~ visit, data = toy))                       # 환자를 무시한 일원분산분석

### 셀 2. 네 번 모두 잰 75명의 반복측정 분산분석

In [ ]:
suppressPackageStartupMessages({library(tidyr); library(dplyr); library(afex)})
w <- read.csv("https://socialp-ajou.tecentriq12.workers.dev/data/ch10_hba1c.csv")                        # 환자 한 명이 한 줄 (넓은 형식)
cc <- w |> filter(!is.na(hba1c_0), !is.na(hba1c_3), !is.na(hba1c_6), !is.na(hba1c_12)) |>
  pivot_longer(starts_with("hba1c_"), names_to = "month", names_prefix = "hba1c_", values_to = "hba1c") |>
  mutate(month = factor(month, levels = c("0", "3", "6", "12")), group = factor(group))
table(cc$group) / 4                                       # 네 번 모두 잰 환자 (군별)
a <- aov_ez("id", "hba1c", cc, between = "group", within = "month", type = 3)
summary(a)                                                # 구형성 가정, Mauchly 검정, GG·HF 보정

### 셀 3. ε으로 자유도를 줄여 P 다시 읽기

In [ ]:
F <- 3.946; eps <- 0.809                                  # 교호작용의 F와 Greenhouse–Geisser ε
round(c(sphericity = pf(F, 3, 219, lower.tail = FALSE),
        GG = pf(F, 3 * eps, 219 * eps, lower.tail = FALSE),
        lower_bound = pf(F, 1, 73, lower.tail = FALSE)), 4)
nice(a, correction = "GG")                                # 보정한 자유도를 함께 적은 표

## b. 넓은 형식과 긴 형식, 결측 양상

### 셀 4. 넓은 형식을 긴 형식으로

In [ ]:
L <- w |> pivot_longer(starts_with("hba1c_"), names_to = "month", names_prefix = "hba1c_",
                       values_to = "hba1c", names_transform = list(month = as.integer)) |>
  select(id, group, month, hba1c)
nrow(L); sum(is.na(L$hba1c))                              # 100명 × 4 = 400행, 그 가운데 빈 값
L <- filter(L, !is.na(hba1c))                             # 측정하지 못한 방문의 행은 지움
nrow(L); head(L, 6)

### 셀 5. 결측 양상과 탈락자의 기저값

In [ ]:
pat <- apply(!is.na(w[, c("hba1c_0", "hba1c_3", "hba1c_6", "hba1c_12")]), 1,
             function(o) paste(ifelse(o, "O", "-"), collapse = ""))
table(pattern = pat, group = w$group)                     # O = 잰 방문, - = 빈 방문
has12 <- !is.na(w$hba1c_12)
aggregate(hba1c_0 ~ group + has12, data = cbind(w, has12), FUN = function(x) round(c(n = length(x), mean = mean(x)), 2))
m12 <- glm(!has12 ~ group + I(hba1c_0 - mean(hba1c_0)), family = binomial, data = w)
round(exp(cbind(OR = coef(m12), confint.default(m12)))[-1, ], 2)   # 12개월 값이 빌 오즈

## c. 선형 혼합모형과 MMRM

### 셀 6. 무작위 절편 모형

In [ ]:
suppressPackageStartupMessages(library(lmerTest))       # lme4의 lmer()에 Satterthwaite 자유도를 더함
ri <- lmer(hba1c ~ group * month + (1 | id), data = L)  # 무작위 절편, REML이 기본값
summary(ri)$coefficients
as.data.frame(VarCorr(ri))[, c("grp", "vcov", "sdcor")]

### 셀 7. 무작위 절편 + 기울기 모형

In [ ]:
rs <- lmer(hba1c ~ group * month + (month | id), data = L,
           control = lmerControl(optimizer = "bobyqa"))  # 무작위 절편 + 기울기
summary(rs)$coefficients
as.data.frame(VarCorr(rs))[, c("var1", "var2", "vcov", "sdcor")]
contest(rs, c(0, 0, 0, 12), joint = FALSE)               # 12개월 군간 차이 = 12 × 군 × 개월
c(nobs = nobs(rs), patients = ngrps(rs))

### 셀 8. 무작위 기울기가 필요한가

In [ ]:
anova(ri, rs, refit = FALSE)                             # 같은 고정효과: REML 우도로 비교
c(REML_ri = REMLcrit(ri), REML_rs = REMLcrit(rs))        # −2 × REML 로그우도

### 셀 9. 공분산 구조 비교

In [ ]:
suppressPackageStartupMessages(library(nlme))
L$fmonth <- factor(L$month); L$vn <- as.integer(L$fmonth)
cov_fit <- function(cor, w = NULL)
  gls(hba1c ~ fmonth * group, data = L, correlation = cor, weights = w, method = "REML")
fits <- list(CS  = cov_fit(corCompSymm(form = ~ 1 | id)),
             AR1 = cov_fit(corAR1(form = ~ vn | id)),
             UN  = cov_fit(corSymm(form = ~ vn | id), varIdent(form = ~ 1 | fmonth)))
t(sapply(fits, function(f) c(m2REML = -2 * as.numeric(logLik(f)), AIC = AIC(f),
                             diff12 = unname(coef(f)["fmonth12:group"]))))

### 셀 10. MMRM: 비구조 공분산과 Satterthwaite 자유도

In [ ]:
suppressWarnings(suppressPackageStartupMessages({library(mmrm); library(emmeans)}))
C <- w |> pivot_longer(c(hba1c_3, hba1c_6, hba1c_12), names_to = "visit", names_prefix = "hba1c_",
                       values_to = "y") |>
  filter(!is.na(y)) |>
  mutate(chg = y - hba1c_0, visit = factor(visit, levels = c("3", "6", "12")),
         group = factor(group), id = factor(id))
C$cbase <- C$hba1c_0 - mean(w$hba1c_0[w$id %in% C$id])   # 기저값을 분석 환자 평균으로 중심화
c(patients = nlevels(droplevels(C$id)), rows = nrow(C))
m <- mmrm(chg ~ visit + group:visit + cbase:visit + us(visit | id), data = C,
          method = "Satterthwaite")                       # 비구조 공분산, REML
em <- emmeans(m, ~ group | visit, at = list(cbase = 0))
em
contrast(em, "revpairwise")                              # 방문마다 중재군 − 일반 진료군

### 셀 11. 결측을 다루는 방법에 따라 달라지는 12개월 차이

In [ ]:
w$chg12 <- w$hba1c_12 - w$hba1c_0
w$cb <- w$hba1c_0 - mean(w$hba1c_0)
cc12 <- lm(chg12 ~ group + cb, data = w)                 # 12개월 값이 있는 환자만 (79명)
ccall <- lm(chg12 ~ group + cb, data = subset(w, !is.na(hba1c_3) & !is.na(hba1c_6)))  # 네 번 모두 잰 75명
locf <- w |> mutate(last = coalesce(hba1c_12, hba1c_6, hba1c_3)) |> filter(!is.na(last))
lo <- lm(I(last - hba1c_0) ~ group + cb, data = locf)    # 마지막 값을 끌어온 분석 (92명)
round(rbind(completers12 = coef(summary(cc12))["group", 1:2], completers_all = coef(summary(ccall))["group", 1:2],
            LOCF = coef(summary(lo))["group", 1:2]), 3)

## d. GEE와 로지스틱 혼합모형

### 셀 12. 순응 자료를 긴 형식으로

In [ ]:
A <- w |> pivot_longer(starts_with("adh_"), names_to = "visit", names_prefix = "adh_", values_to = "adh") |>
  filter(!is.na(adh)) |> mutate(visit = factor(visit, levels = c("3", "6", "12"))) |>
  arrange(id, visit)                                     # GEE는 같은 환자의 행이 붙어 있어야 함
c(rows = nrow(A), patients = length(unique(A$id)))
with(A, round(tapply(adh, list(group, visit), mean), 3)) # 기간별 순응 비율 (행: 일반 진료 0, 중재 1)

### 셀 13. 교환가능 작업상관 GEE

In [ ]:
suppressPackageStartupMessages(library(geepack))
ge <- geeglm(adh ~ group + visit, id = id, data = A, family = binomial, corstr = "exchangeable")
summary(ge)
b <- coef(ge)["group"]; se <- summary(ge)$coefficients["group", "Std.err"]
round(exp(c(OR = b, b - 1.96 * se, b + 1.96 * se)), 2)

### 셀 14. 작업상관 네 가지와 상관을 무시한 표준오차

In [ ]:
t(sapply(c("independence", "exchangeable", "ar1", "unstructured"), function(cs) {
  g <- geeglm(adh ~ group + visit, id = id, waves = as.integer(visit), data = A,
              family = binomial, corstr = cs)
  s <- summary(g)$coefficients["group", "Std.err"]
  round(c(OR = exp(coef(g)[["group"]]), lower = exp(coef(g)[["group"]] - 1.96 * s),
          upper = exp(coef(g)[["group"]] + 1.96 * s), robust_SE = s, QIC = QIC(g)[["QIC"]]), 3)
}))
gl0 <- glm(adh ~ group + visit, family = binomial, data = A)   # 같은 환자의 상관을 무시하면
round(summary(gl0)$coefficients["group", 1:2], 4)

### 셀 15. 로지스틱 혼합모형의 조건부 오즈비

In [ ]:
suppressPackageStartupMessages(library(lme4))
lap <- glmer(adh ~ group + visit + (1 | id), data = A, family = binomial)              # 라플라스 근사
quad <- glmer(adh ~ group + visit + (1 | id), data = A, family = binomial, nAGQ = 40)   # 적분점 40개
se <- sqrt(vcov(quad)["group", "group"])
round(rbind(Laplace = c(OR = exp(fixef(lap)[["group"]]), SD = attr(VarCorr(lap)$id, "stddev")[[1]], NA, NA),
            quadrature = c(exp(fixef(quad)[["group"]]), attr(VarCorr(quad)$id, "stddev")[[1]],
                           exp(fixef(quad)[["group"]] + c(-1, 1) * 1.96 * se))), 2)

## e. 과제 정답

### 셀 16. 과제 1 정답

In [ ]:
pf(18.044, 3 * 0.8087, 219 * 0.8087, lower.tail = FALSE)   # 시간 주효과, GG 보정
pf(3.946, 1, 73, lower.tail = FALSE)                       # 교호작용, 하한 보정

### 셀 17. 과제 2 정답

In [ ]:
gi <- geeglm(adh ~ group * visit, id = id, data = A, family = binomial, corstr = "exchangeable")
anova(ge, gi)                                             # 군 × 기간 곱셈항 2개의 Wald 검정